# TraderPro - Reinforcement Learning (Kaggle Version)

Entraînement d'un agent RL pour le trading Bitcoin.

## Configuration
- **Actions**: Discrètes (BUY / SELL / HOLD)
- **Horizon**: Daily
- **Objectif**: Sharpe Ratio
- **Algorithme**: PPO (Stable-Baselines3)
- **Data Source**: Twelve Data API (3 ans d'historique)

## Kaggle Setup
- Les fichiers seront sauvegardés dans `/kaggle/working/`
- Téléchargez-les depuis l'onglet **Output** après exécution

---
## 1. Installation des dépendances

In [ ]:
!pip install stable-baselines3 gymnasium --quiet
print("Installation terminée!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from datetime import datetime, timedelta
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Reinforcement Learning
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, DQN, A2C
from stable_baselines3.common.callbacks import BaseCallback

# Kaggle output directory
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Imports OK!")
print(f"Output directory: {OUTPUT_DIR}")

---
## 2. Récupération des données Twelve Data API

L'API Binance peut être bloquée sur certains serveurs, on utilise Twelve Data comme alternative fiable.

In [ ]:
# Data loader - Binance 1h candles with fallback to Kaggle CSV
import time, os
BINANCE_INTERVAL_MS = {
    "1m": 60_000,
    "5m": 300_000,
    "15m": 900_000,
    "1h": 3_600_000,
    "4h": 14_400_000,
    "1d": 86_400_000,
}
BINANCE_HOSTS = ["https://api.binance.com", "https://api.binance.us"]
FALLBACK_CSV = "/kaggle/input/binance-btcusdt-1h/binance_BTCUSDT_1h.csv"  # change si dataset différent

def fetch_binance(symbol="BTCUSDT", interval="1h", lookback_days=730):
    """Télécharge les chandelles OHLCV via l'API publique Binance (multi-host)."""
    ms = BINANCE_INTERVAL_MS[interval]
    end_time = int(time.time() * 1000)
    start_time = end_time - lookback_days * 24 * 60 * 60 * 1000
    frames = []
    while start_time < end_time:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": start_time,
            "endTime": min(start_time + ms * 1000, end_time),
            "limit": 1000,
        }
        last_error = None
        batch = None
        for host in BINANCE_HOSTS:
            url = f"{host}/api/v3/klines"
            resp = requests.get(url, params=params, timeout=30)
            if resp.status_code == 451:
                last_error = Exception(f"{host} renvoie 451 (region blocked)")
                continue
            try:
                resp.raise_for_status()
                batch = resp.json()
                break
            except Exception as e:
                last_error = e
        if last_error and batch is None:
            raise last_error

        if not batch:
            break
        frames.extend(batch)
        start_time = batch[-1][0] + ms
        time.sleep(0.2)  # évite le rate limiting

    cols = ["open_time","open","high","low","close","volume","close_time","q","n","tb","tq","ignore"]
    df = pd.DataFrame(frames, columns=cols)
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms")
    df = (
        df.set_index("timestamp")[["open","high","low","close","volume"]]
          .astype(float)
          .sort_index()
          .rename_axis(None)
    )
    print(f"Binance data: {len(df)} candles ({df.index[0]} → {df.index[-1]})")
    return df

def load_price_data():
    try:
        return fetch_binance("BTCUSDT", "1h", lookback_days=730)
    except Exception as e:
        print(f"⚠️ Binance API inaccessible ({e}). Fallback vers CSV local.")
        if not os.path.exists(FALLBACK_CSV):
            raise
        df = pd.read_csv(FALLBACK_CSV)
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        elif 'open_time' in df.columns:
            df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms')
        else:
            raise RuntimeError('CSV fallback must contain timestamp or open_time column')
        df = (
            df.set_index('timestamp')[['open','high','low','close','volume']]
              .astype(float)
              .sort_index()
        )
        print(f"CSV fallback: {len(df)} chandelles ({df.index[0]} → {df.index[-1]})")
        return df

raw_df = load_price_data()
raw_df.head()


---
## 3. Feature Engineering

In [ ]:
def create_features(df):
    """Crée les features techniques pour l'agent RL."""
    data = df.copy()
    
    # Returns
    data['returns'] = data['close'].pct_change()
    data['returns_2d'] = data['close'].pct_change(2)
    data['returns_5d'] = data['close'].pct_change(5)
    data['returns_10d'] = data['close'].pct_change(10)
    
    # Volatilité
    data['volatility_7d'] = data['returns'].rolling(7).std()
    data['volatility_14d'] = data['returns'].rolling(14).std()
    
    # RSI
    delta = data['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-10)
    data['rsi'] = 100 - (100 / (1 + rs))
    data['rsi_normalized'] = data['rsi'] / 100
    
    # MACD
    ema12 = data['close'].ewm(span=12).mean()
    ema26 = data['close'].ewm(span=26).mean()
    data['macd'] = ema12 - ema26
    data['macd_signal'] = data['macd'].ewm(span=9).mean()
    data['macd_hist'] = data['macd'] - data['macd_signal']
    data['macd_normalized'] = data['macd_hist'] / (data['close'] + 1e-10)
    
    # Bollinger Bands
    sma20 = data['close'].rolling(20).mean()
    std20 = data['close'].rolling(20).std()
    data['bb_upper'] = sma20 + 2 * std20
    data['bb_lower'] = sma20 - 2 * std20
    data['bb_position'] = (data['close'] - data['bb_lower']) / (data['bb_upper'] - data['bb_lower'] + 1e-10)
    
    # Moving Averages
    data['sma_7'] = data['close'].rolling(7).mean()
    data['sma_25'] = data['close'].rolling(25).mean()
    data['price_sma7_ratio'] = data['close'] / (data['sma_7'] + 1e-10)
    data['price_sma25_ratio'] = data['close'] / (data['sma_25'] + 1e-10)
    data['sma7_sma25_ratio'] = data['sma_7'] / (data['sma_25'] + 1e-10)
    
    # Volume
    data['volume_sma'] = data['volume'].rolling(20).mean()
    data['volume_ratio'] = data['volume'] / (data['volume_sma'] + 1e-10)
    
    # Range
    data['daily_range'] = (data['high'] - data['low']) / (data['close'] + 1e-10)
    
    # Clean
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna()
    
    return data

df_features = create_features(raw_df)
print(f"Features créées: {len(df_features)} lignes, {len(df_features.columns)} colonnes")

---
## 4. Environnement de Trading

In [ ]:
class TradingEnvironment(gym.Env):
    """
    Environnement de trading pour RL.
    
    Actions: 0=HOLD, 1=BUY, 2=SELL
    Reward: Sharpe ratio based
    """
    
    metadata = {'render_modes': ['human']}
    
    def __init__(self, df, initial_capital=10000, transaction_fee=0.001, window_size=14):
        super().__init__()
        
        self.df = df.reset_index(drop=True)
        self.initial_capital = initial_capital
        self.transaction_fee = transaction_fee
        self.window_size = window_size
        
        self.feature_columns = [
            'returns', 'returns_2d', 'returns_5d', 'returns_10d',
            'volatility_7d', 'volatility_14d',
            'rsi_normalized', 'macd_normalized', 'bb_position',
            'price_sma7_ratio', 'price_sma25_ratio', 'sma7_sma25_ratio',
            'volume_ratio', 'daily_range'
        ]
        
        self.action_space = spaces.Discrete(3)  # HOLD, BUY, SELL
        
        n_features = len(self.feature_columns) + 2
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(n_features,), dtype=np.float32
        )
        
        self.reset()
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.current_step = self.window_size
        self.capital = self.initial_capital
        self.position = 0
        self.entry_price = 0
        self.total_profit = 0
        self.trades = []
        self.portfolio_values = [self.initial_capital]
        self.returns_history = []
        self.max_portfolio_value = self.initial_capital
        
        return self._get_observation(), {}
    
    def _get_observation(self):
        features = self.df.loc[self.current_step, self.feature_columns].values
        
        unrealized_pnl = 0
        if self.position == 1:
            current_price = self.df.loc[self.current_step, 'close']
            unrealized_pnl = (current_price - self.entry_price) / self.entry_price
        
        obs = np.concatenate([features, [self.position, unrealized_pnl]]).astype(np.float32)
        obs = np.nan_to_num(obs, nan=0.0, posinf=0.0, neginf=0.0)
        
        return obs
    
    def step(self, action):
        current_price = self.df.loc[self.current_step, 'close']
        reward = 0
        trade_profit = 0
        
        # Execute action
        if action == 1 and self.position == 0:  # BUY
            self.position = 1
            self.entry_price = current_price
            self.capital *= (1 - self.transaction_fee)
                
        elif action == 2 and self.position == 1:  # SELL
            trade_profit = (current_price - self.entry_price) / self.entry_price
            self.capital *= (1 + trade_profit) * (1 - self.transaction_fee)
            self.total_profit += trade_profit
            self.trades.append({'entry': self.entry_price, 'exit': current_price, 'profit': trade_profit})
            self.position = 0
            self.entry_price = 0
        
        # Update portfolio value
        if self.position == 1:
            unrealized = (current_price - self.entry_price) / self.entry_price
            portfolio_value = self.capital * (1 + unrealized)
        else:
            portfolio_value = self.capital
        
        self.portfolio_values.append(portfolio_value)
        self.max_portfolio_value = max(self.max_portfolio_value, portfolio_value)
        
        # Daily return
        daily_return = (portfolio_value - self.portfolio_values[-2]) / self.portfolio_values[-2] if len(self.portfolio_values) > 1 else 0
        self.returns_history.append(daily_return)
        
        # === REWARD (Sharpe-based) ===
        if len(self.returns_history) >= 7:
            recent_returns = np.array(self.returns_history[-7:])
            if np.std(recent_returns) > 0:
                sharpe = np.mean(recent_returns) / np.std(recent_returns) * np.sqrt(252)
                reward += sharpe * 0.1
        
        reward += trade_profit * 10
        
        drawdown = (self.max_portfolio_value - portfolio_value) / self.max_portfolio_value
        if drawdown > 0.1:
            reward -= drawdown * 5
        
        if action == 0 and self.position == 0:
            reward -= 0.001
        
        self.current_step += 1
        
        terminated = self.current_step >= len(self.df) - 1
        truncated = self.capital <= self.initial_capital * 0.5
        
        info = {
            'capital': self.capital,
            'portfolio_value': portfolio_value,
            'position': self.position,
            'total_trades': len(self.trades),
            'drawdown': drawdown
        }
        
        return self._get_observation(), reward, terminated, truncated, info
    
    def get_metrics(self):
        if len(self.portfolio_values) < 2:
            return {}
        
        returns = np.diff(self.portfolio_values) / self.portfolio_values[:-1]
        total_return = (self.portfolio_values[-1] - self.initial_capital) / self.initial_capital
        sharpe = np.mean(returns) / (np.std(returns) + 1e-10) * np.sqrt(252) if len(returns) > 0 else 0
        max_dd = np.max(np.maximum.accumulate(self.portfolio_values) - self.portfolio_values) / np.maximum.accumulate(self.portfolio_values).max()
        
        winning_trades = [t for t in self.trades if t['profit'] > 0]
        win_rate = len(winning_trades) / len(self.trades) if self.trades else 0
        
        return {
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_dd,
            'total_trades': len(self.trades),
            'win_rate': win_rate,
            'final_capital': self.portfolio_values[-1]
        }

# Test
print("Test de l'environnement...")
env = TradingEnvironment(df_features)
obs, _ = env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")

---
## 5. Entraînement PPO

In [ ]:
class TrainingCallback(BaseCallback):
    """Callback pour suivre l'entraînement."""
    
    def __init__(self, eval_env, eval_freq=5000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.best_sharpe = -np.inf
        self.metrics_history = []
    
    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            obs, _ = self.eval_env.reset()
            done = False
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = self.eval_env.step(action)
                done = terminated or truncated
            
            metrics = self.eval_env.get_metrics()
            self.metrics_history.append(metrics)
            
            if self.verbose > 0:
                print(f"\n[Step {self.n_calls}] Return: {metrics.get('total_return', 0)*100:.1f}% | "
                      f"Sharpe: {metrics.get('sharpe_ratio', 0):.2f} | "
                      f"Trades: {metrics.get('total_trades', 0)} | "
                      f"Win: {metrics.get('win_rate', 0)*100:.0f}%")
            
            if metrics.get('sharpe_ratio', 0) > self.best_sharpe:
                self.best_sharpe = metrics.get('sharpe_ratio', 0)
                self.model.save(f"{OUTPUT_DIR}/best_trading_agent")
                print(f"  -> New best model! Sharpe: {self.best_sharpe:.2f}")
        
        return True

In [ ]:
# Split train/test
split_idx = int(len(df_features) * 0.8)
train_df = df_features.iloc[:split_idx].reset_index(drop=True)
test_df = df_features.iloc[split_idx:].reset_index(drop=True)

print(f"Train: {len(train_df)} jours")
print(f"Test: {len(test_df)} jours")

train_env = TradingEnvironment(train_df)
eval_env = TradingEnvironment(test_df)

callback = TrainingCallback(eval_env, eval_freq=5000, verbose=1)

In [ ]:
# Créer le modèle PPO
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=0
)

print("Modèle PPO créé!")

In [ ]:
# === ENTRAÎNEMENT ===
# Ajustez selon vos besoins:
# - 50000: ~5 min (test rapide)
# - 100000: ~10 min (bon compromis)
# - 200000: ~20 min (meilleur)
# - 500000: ~1h (optimal)

TOTAL_TIMESTEPS = 100000

print(f"Début entraînement ({TOTAL_TIMESTEPS:,} timesteps)...")
print("=" * 60)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
    progress_bar=True
)

print("=" * 60)
print("Entraînement terminé!")

---
## 6. Évaluation et Backtesting

In [ ]:
# Charger le meilleur modèle
best_model = PPO.load(f"{OUTPUT_DIR}/best_trading_agent")
print("Meilleur modèle chargé!")

# Backtest sur données de test
test_env = TradingEnvironment(test_df)
obs, _ = test_env.reset()

actions_taken = []
portfolio_history = [test_env.initial_capital]
price_history = [test_df.iloc[test_env.window_size]['close']]

done = False
while not done:
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = test_env.step(action)
    done = terminated or truncated
    
    actions_taken.append(['HOLD', 'BUY', 'SELL'][action])
    portfolio_history.append(info['portfolio_value'])
    if test_env.current_step < len(test_df):
        price_history.append(test_df.iloc[test_env.current_step]['close'])

# Résultats
metrics = test_env.get_metrics()

print("\n" + "=" * 60)
print("         RÉSULTATS BACKTEST (Données de Test)")
print("=" * 60)
print(f"  Return Total:    {metrics['total_return']*100:>10.2f}%")
print(f"  Sharpe Ratio:    {metrics['sharpe_ratio']:>10.2f}")
print(f"  Max Drawdown:    {metrics['max_drawdown']*100:>10.2f}%")
print(f"  Nombre Trades:   {metrics['total_trades']:>10}")
print(f"  Win Rate:        {metrics['win_rate']*100:>10.1f}%")
print(f"  Capital Final:   ${metrics['final_capital']:>10,.2f}")
print("=" * 60)

In [ ]:
# Visualisation
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Portfolio vs Buy & Hold
ax1 = axes[0]
initial_price = price_history[0]
buy_hold = [test_env.initial_capital * (p / initial_price) for p in price_history[:len(portfolio_history)]]

ax1.plot(portfolio_history, label='Agent RL', color='green', linewidth=2)
ax1.plot(buy_hold, label='Buy & Hold', color='blue', linestyle='--', alpha=0.7)
ax1.axhline(y=test_env.initial_capital, color='gray', linestyle=':', alpha=0.5)
ax1.set_title('Performance: Agent RL vs Buy & Hold', fontsize=14, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Prix avec signaux
ax2 = axes[1]
ax2.plot(price_history[:len(actions_taken)], label='BTC Price', color='orange', alpha=0.7)

for i, action in enumerate(actions_taken):
    if i < len(price_history):
        if action == 'BUY':
            ax2.scatter(i, price_history[i], color='green', marker='^', s=100, zorder=5)
        elif action == 'SELL':
            ax2.scatter(i, price_history[i], color='red', marker='v', s=100, zorder=5)

ax2.set_title('BTC Price with Trading Signals', fontsize=14, fontweight='bold')
ax2.set_ylabel('Price ($)')
ax2.legend(['BTC', 'BUY', 'SELL'])
ax2.grid(True, alpha=0.3)

# 3. Distribution des actions
ax3 = axes[2]
action_counts = pd.Series(actions_taken).value_counts()
colors = {'HOLD': 'gray', 'BUY': 'green', 'SELL': 'red'}
bars = ax3.bar(action_counts.index, action_counts.values, 
               color=[colors.get(a, 'blue') for a in action_counts.index])
ax3.set_title('Distribution des Actions', fontsize=14, fontweight='bold')
ax3.set_ylabel('Count')

for bar, count in zip(bars, action_counts.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             str(count), ha='center', fontsize=12)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rl_backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nGraphique sauvegardé: {OUTPUT_DIR}/rl_backtest_results.png")

---
## 7. Sauvegarde pour TraderPro

In [ ]:
# Sauvegarder le modèle final
model.save(f"{OUTPUT_DIR}/trading_agent_ppo_final")

# Configuration
config = {
    "model_type": "PPO",
    "model_version": "1.0.0",
    "trained_at": datetime.now().isoformat(),
    "training_timesteps": TOTAL_TIMESTEPS,
    "feature_columns": [
        'returns', 'returns_2d', 'returns_5d', 'returns_10d',
        'volatility_7d', 'volatility_14d',
        'rsi_normalized', 'macd_normalized', 'bb_position',
        'price_sma7_ratio', 'price_sma25_ratio', 'sma7_sma25_ratio',
        'volume_ratio', 'daily_range'
    ],
    "actions": ["HOLD", "BUY", "SELL"],
    "action_space": "Discrete(3)",
    "initial_capital": 10000,
    "transaction_fee": 0.001,
    "metrics": {
        "total_return": float(metrics['total_return']),
        "sharpe_ratio": float(metrics['sharpe_ratio']),
        "max_drawdown": float(metrics['max_drawdown']),
        "total_trades": int(metrics['total_trades']),
        "win_rate": float(metrics['win_rate'])
    }
}

with open(f"{OUTPUT_DIR}/training_config_rl.json", "w") as f:
    json.dump(config, f, indent=2)

print("="*60)
print("FICHIERS SAUVEGARDÉS DANS /kaggle/working/")
print("="*60)
print(f"  1. trading_agent_ppo_final.zip  (modèle)")
print(f"  2. training_config_rl.json      (config)")
print(f"  3. rl_backtest_results.png      (graphique)")
print(f"  4. best_trading_agent.zip       (meilleur modèle)")
print("="*60)

# Lister les fichiers
print("\nFichiers disponibles:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"  - {f} ({size/1024:.1f} KB)")

---
## 8. Test de Prédiction

In [ ]:
def predict_action(model, recent_data, position=0, unrealized_pnl=0):
    """
    Prédit l'action à prendre (simulation backend).
    
    Args:
        model: Modèle PPO chargé
        recent_data: DataFrame avec données récentes (min 60 jours)
        position: Position actuelle (0=none, 1=long)
        unrealized_pnl: PnL non réalisé
    
    Returns:
        dict avec action et métadonnées
    """
    features_df = create_features(recent_data)
    
    feature_columns = [
        'returns', 'returns_2d', 'returns_5d', 'returns_10d',
        'volatility_7d', 'volatility_14d',
        'rsi_normalized', 'macd_normalized', 'bb_position',
        'price_sma7_ratio', 'price_sma25_ratio', 'sma7_sma25_ratio',
        'volume_ratio', 'daily_range'
    ]
    
    last_features = features_df[feature_columns].iloc[-1].values
    obs = np.concatenate([last_features, [position, unrealized_pnl]]).astype(np.float32)
    obs = np.nan_to_num(obs, nan=0.0)
    
    action, _ = model.predict(obs, deterministic=True)
    
    action_names = ["HOLD", "BUY", "SELL"]
    
    return {
        "action": action_names[action],
        "action_code": int(action),
        "current_price": float(recent_data['close'].iloc[-1]),
        "rsi": float(features_df['rsi'].iloc[-1]),
        "timestamp": datetime.now().isoformat()
    }

# Test
print("Test de prédiction avec données actuelles:")
print("=" * 50)
result = predict_action(best_model, df.tail(60))
print(f"  Action:      {result['action']}")
print(f"  Prix:        ${result['current_price']:,.2f}")
print(f"  RSI:         {result['rsi']:.1f}")
print(f"  Timestamp:   {result['timestamp']}")

---
## Instructions pour TraderPro

### Télécharger les fichiers
1. Cliquez sur l'onglet **Output** (à droite)
2. Téléchargez:
   - `trading_agent_ppo_final.zip`
   - `training_config_rl.json`

### Installation dans TraderPro
```bash
# Copier dans le backend
cp trading_agent_ppo_final.zip backend/app/ml/models/
cp training_config_rl.json backend/app/ml/models/
```

### Utilisation API
```
GET /api/rl/action?symbol=BTCUSDT
GET /api/rl/status
```